# 📈 JP Backtest — Architecture & Component Walkthrough

Welcome to the interactive guide to understanding the **Japan Stock Backtesting App**. We will walk through the entire lifecycle of a single data point from the **Yahoo Finance API**, all the way linearly to the **React dashboard frontend**.

***Note:*** *You can actively run the code cells in this notebook by clicking on them and pressing `Shift + Enter`.*

## A: Python Producer to retrieve yfinance (`yfinance` ➔ `Producer`)
The journey begins with the **Data Pipeline**. A Python script periodically polls the `yfinance` library for end-of-day market data (Open, High, Low, Close, Volume) of Japanese blue-chip stocks.

Let's test this strictly right now by pulling Apple's data locally:

In [ ]:
import yfinance as yf
import pandas as pd

# The Python Producer pulls raw data
print("Fetching live data for Toyota (7203.T)...")
ticker = yf.Ticker('7203.T')
history = ticker.history(period='5d')
display(history[['Open', 'High', 'Low', 'Close', 'Volume']].tail())

Instead of writing this data directly to a database, we package this payload into a serialized message and dispatch it to a messaging queue to prevent blockages during high-frequency data pulls.

## B: Run BlazingMQ with Python Consumer (`Producer` ➔ `BlazingMQ` ➔ `Consumer`)
To introduce decoupled resilience, the producer drops the payload onto **BlazingMQ**, Bloomberg's open-source highly-performant message queue.
- **Producer**: Broadcasts unstructured OHLCV JSON payloads into a queue cluster.
- **Broker (BlazingMQ)**: Persists the data securely.
- **Consumer**: A completely separate Python background service listens to this queue, acknowledges receipt of the raw data, and flushes it securely to persistent batch storage (e.g., Parquet or JSON formats) for Data Engineering.

## C: How Data is Fed to Databricks (`Consumer` ➔ `Delta Lake`)
The batched data outputted by the Consumer is synchronized into a **Databricks Delta Lake**.

Inside Databricks, Apache Spark executes parallel transformations on this Data Lake. Let's simulate that math transformation here natively to calculate an indicator like a Moving Average:

In [ ]:
import numpy as np

# Simulate the Databricks Spark DataFrame calculating a 3-day Moving Average
df = history.copy()
df['SMA_3'] = df['Close'].rolling(window=3).mean()

print("Engineered Features (Ready for ML):")
display(df[['Close', 'SMA_3']].tail(5))

Databricks successfully refines the "raw data" into highly structured, mathematically complete "feature data" which forms the ground truth for our application.

## D: How AWS SageMaker Works with Data in Databricks (`Databricks` ➔ `AWS ML`)
With the "feature data" fully compiled (including SMA, RSI, and Bollinger variables) in Databricks, we tap into **AWS SageMaker**.

1. **Training Mode**: Historical feature data is securely piped from Databricks into SageMaker. An **XGBoost Classifier Model** runs extensive cross-validation routines to train a backtesting simulation evaluating our custom algorithmic strategies against historical trends.

Let's simulate a simplified quantitative algorithmic backtest right here using `quantstats`:

In [ ]:
import quantstats as qs
import warnings
warnings.filterwarnings('ignore')

# Extend the dataset to 1 Year for a better Backtest simulation
test_data = yf.download('7203.T', period='1y', progress=False)
close_prices = test_data['Close']
if isinstance(close_prices, pd.DataFrame):
    close_prices = close_prices.iloc[:, 0]
close_prices = close_prices.squeeze()

# Simple Strategy: Buy & Hold (Daily Returns)
daily_returns = close_prices.pct_change().dropna()

# Calculate AWS Model metrics metrics
sharpe = qs.stats.sharpe(daily_returns)
cagr = qs.stats.cagr(daily_returns)
win_rate = qs.stats.win_rate(daily_returns)

print(f" Simulated Backend ML Evaluation Metrics:")
print(f" 📊 Sharpe Ratio: {sharpe:.2f}")
print(f" 📈 CAGR: {cagr*100:.2f}%")
print(f" 🏆 Win Rate: {win_rate*100:.2f}%")

print("\n2. Inference / Real-Time: Utilizing the finalized model from above, newly ingested daily data vectors are passed from the Databricks pipeline to the SageMaker InvokeEndpoint API to obtain active predictions (decisions like BUY, SELL, HOLD).")

## E: How Java Spring Boot Interacts with AWS (`SageMaker` + `Databricks` ➔ `Spring Boot`)
Our **Java Spring Boot API** acts as the secure middleman of this entire application. 

The backend loads the combined state:
1. The finalized market numbers originating from the Databricks layer.
2. The associated machine-learning backtest decisions retrieved from AWS.

```java
@RestController
public class StockController {
    @GetMapping("/api/stocks")
    public List<StockBacktest> getAllBacktests() {
        // Returns the strictly-typed aggregate composed of data & model AI decisions
        return repository.findAll();
    }
}
```
Java handles object serialization, data persistence caching, and protects downstream clients by delivering only perfectly shaped JSON APIs.

## F: How React Works with the Java Backend's Output (`Spring Boot` ➔ `React Vite`)
The final stop is the user's browser. Our dynamic **React 18 + Vite** frontend launches a performant Single-Page Application (SPA).

Using `axios` directly within `useEffect` hooks, the frontend queries the `/api/stocks` Java REST endpoint, retrieves the typed array of `StockBacktest` models, and passes them as state:
```javascript
// Data ingestion from backend
axios.get('http://localhost:8080/api/stocks').then((res) => setStocks(res.data));

// Rendered to screen smoothly mapped to UI components
stocks.map((stock) => <BacktestCard data={stock} />)
```
The UI relies exclusively on data pre-calculated by Databricks, securely signed by AWS SageMaker, and safely hosted by the Java middleware, ensuring the client application remains extraordinarily fast and absolutely secure.